# Lab | Classification

In this lab, you'll work with the Palmer Penguins dataset to train and compare multiple classification algorithms. You'll evaluate models using precision, recall, F1 score, confusion matrices, and ROC curves.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_curve, auc,
                             ConfusionMatrixDisplay, classification_report)

sns.set_style("whitegrid")

## Task 1: Data Prep & Baseline

### 1. Load the dataset and drop missing values

In [ ]:
penguins = sns.load_dataset("penguins").dropna()
penguins.head()

### 2. Brief Exploration

In [ ]:
print(f"Samples: {penguins.shape[0]}, Features: {penguins.shape[1] - 1}")
print("\nSpecies Distribution:")
print(penguins['species'].value_counts())
print("\nFeature Types:")
print(penguins.dtypes)

### 3. Prepare Features

In [ ]:
# Encode the target (species)
le = LabelEncoder()
y = le.fit_transform(penguins['species'])

# Encode categorical features (island, sex)
X = penguins.drop('species', axis=1)
X = pd.get_dummies(X, columns=['island', 'sex'], drop_first=True)

### 4. Split and Scale

In [ ]:
# Split into training and test sets (80/20, stratify=y, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

# Scale the features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 5. Logistic Regression Baseline

In [ ]:
log_reg = LogisticRegression(max_iter=10000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

y_pred = log_reg.predict(X_test_scaled)

print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

### Interpretation of Task 1 Results

The Logistic Regression model provides a very strong baseline, likely achieving over 95% accuracy. Typically:
- **Gentoo** penguins are easiest to classify because they are physically larger and have distinct bill/flipper measurements compared to the other two species.
- **Adelie** and **Chinstrap** can be harder to distinguish since they are closer in size, though they still have notable differences in bill shape.

## Task 2: Algorithm Comparison

In [ ]:
models = {
    "GaussianNB": GaussianNB(),
    "SVC (Linear)": SVC(kernel="linear", probability=True, random_state=42),
    "SVC (RBF)": SVC(kernel="rbf", probability=True, random_state=42),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1": f1_score(y_test, y_pred, average='weighted')
    })

comparison_df = pd.DataFrame(results).sort_values(by="F1", ascending=False)
comparison_df

### Discussion of Task 2 Results

The **RandomForest** and **SVC (Linear/RBF)** models usually perform exceptionally well on this dataset. RandomForest handles non-linear relationships and interactions between features well, while SVC is robust for high-dimensional classification. Decision trees might slightly overfit, and Naive Bayes might underperform if features are highly correlated.

## Task 3: Confusion Matrices & ROC Curves

In [ ]:
# Plot Confusion Matrices
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

all_models = {"LogisticRegression": log_reg, **models}

for i, (name, model) in enumerate(all_models.items()):
    ConfusionMatrixDisplay.from_estimator(model, X_test_scaled, y_test, display_labels=le.classes_, ax=axes[i], cmap='Blues')
    axes[i].set_title(name)

plt.tight_layout()
plt.show()

In [ ]:
# Plot ROC Curves (One-vs-Rest)
from sklearn.preprocessing import label_binarize

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
n_classes = y_test_bin.shape[1]

plt.figure(figsize=(10, 8))

for name, model in all_models.items():
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test_scaled)
    else:
        y_score = model.decision_function(X_test_scaled)
        
    # Compute ROC curve and ROC area for each class
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    
    # Average AUC for all classes
    mean_auc = np.mean(list(roc_auc.values()))
    
    # Macro-average ROC curve
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    
    plt.plot(all_fpr, mean_tpr, label=f'{name} (AUC = {mean_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (Macro-Average)')
plt.legend(loc="lower right")
plt.show()

### Discussion of Task 3 Visualizations

- **Confusion Matrices**: Often show confusion between Adelie and Chinstrap. These two species share more physical similarities compared to the larger Gentoo penguins.
- **ROC Curves**: High AUC values across most models indicate that the features are highly discriminative. RandomForest and Logistic Regression often show nearly perfect AUC.
- **Recommendation**: Based on the metrics and stability, **RandomForest** or **SVC** is typically recommended for this dataset.

## Task 4: Hyperparameter Exploration

### 1. Tune the best-performing model (RandomForest assumed)

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10]
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring="f1_weighted")
grid_search.fit(X_train_scaled, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV F1 Score:", grid_search.best_score_)

### 2. Evaluate Tuned Model

In [ ]:
best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test_scaled)

print("Tuned RandomForest Classification Report:")
print(classification_report(y_test, y_pred_tuned, target_names=le.classes_))

### Reflection on Task 4

- **Improvement**: On such a clean dataset, hyperparameter tuning might only yield marginal improvements, as the default parameters already perform very well.
- **Overfitting**: 5-fold cross-validation helps mitigate overfitting to specific validation folds.
- **Impact**: Tuning is most impactful on noisy, complex datasets or when the default model is underperforming/overfitting significantly.